# ERA5-Land Download — All Spain PV Plants

Downloads ERA5-Land reanalysis (0.1° / ~10km resolution) for all 593 plants via the **CDS Timeseries API**.

**API endpoint:** `reanalysis-era5-land-timeseries` — optimized for single-point time series retrieval (ARCO Zarr backend). Accepts full 3-year date ranges in a single request, unlike the standard ERA5-Land endpoint which has strict per-request size limits.

**Variables:** `t2m`, `d2m`, `sp`, `tp`, `ssrd`, `strd` (all 6 — including longwave thermal radiation)

**Grid point deduplication:** 593 plants snap to 386 unique 0.1° grid points. Each grid point is downloaded once and copied to all plants that share it.

**Estimated runtime:** 386 requests × ~30s each = ~3 hours

**Output:** `era5_timeseries_plants/plant_XXX/plant_XXX_combined.csv`

In [9]:
import os
import time
import shutil
import zipfile
import pandas as pd
import cdsapi
from pathlib import Path
from tqdm.auto import tqdm

if os.path.basename(os.getcwd()) != 'spain_total':
    candidate = os.path.join(os.getcwd(), 'spain_total')
    if os.path.isdir(candidate):
        os.chdir(candidate)
print(f'Working directory: {os.getcwd()}')

Working directory: c:\Users\nicol\Documents\ML-solar-forecast\spain_total


In [10]:
META_FILE  = 'data/plant_metadata.csv'
ERA5_DIR   = Path('era5_timeseries_plants')
CACHE_DIR  = Path('era5_gridpoint_cache')

START_DATE = '2023-01-01'
END_DATE   = '2025-12-31'

CDS_VARIABLES = [
    '2m_temperature',
    '2m_dewpoint_temperature',
    'surface_pressure',
    'total_precipitation',
    'surface_solar_radiation_downwards',
    'surface_thermal_radiation_downwards',
]

SLEEP_BETWEEN = 2.0

In [11]:
meta = pd.read_csv(META_FILE)
print(f'Total plants: {len(meta)}')

# Snap to ERA5-Land 0.1° grid
meta['grid_lat'] = meta['latitude'].round(1)
meta['grid_lon'] = meta['longitude'].round(1)

unique_gp = meta[['grid_lat', 'grid_lon']].drop_duplicates().reset_index(drop=True)
print(f'Unique 0.1° grid points: {len(unique_gp)}')
print(f'Requests saved by dedup: {len(meta) - len(unique_gp)}')

Total plants: 593
Unique 0.1° grid points: 386
Requests saved by dedup: 207


## Step 1 — Download ERA5 per unique grid point

In [12]:
ERA5_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

client = cdsapi.Client()


def cache_path(lat, lon):
    lat_s = f'{lat:.1f}'.replace('-', 'm').replace('.', 'p')
    lon_s = f'{lon:.1f}'.replace('-', 'm').replace('.', 'p')
    return CACHE_DIR / f'{lat_s}_{lon_s}.csv'


def download_gridpoint(lat, lon):
    """Download ERA5-Land timeseries for a single grid point. Merges all CSVs from zip."""
    out_csv = cache_path(lat, lon)
    if out_csv.exists():
        return out_csv

    tmp_zip = out_csv.with_suffix('.zip')

    for attempt in range(5):
        try:
            client.retrieve(
                'reanalysis-era5-land-timeseries',
                {
                    'variable': CDS_VARIABLES,
                    'location': {'latitude': lat, 'longitude': lon},
                    'date': [f'{START_DATE}/{END_DATE}'],
                    'data_format': 'csv',
                },
                str(tmp_zip),
            )
            break
        except Exception as e:
            if '429' in str(e) or 'Too Many' in str(e):
                wait = 120 * (attempt + 1)
                print(f'    Rate limited (attempt {attempt+1}) — waiting {wait}s')
                time.sleep(wait)
            else:
                raise
    else:
        raise RuntimeError(f'Failed after 5 retries for ({lat}, {lon})')

    # The zip contains 3 CSVs (temp, radiation, pressure) — merge them
    with zipfile.ZipFile(tmp_zip) as z:
        csv_names = [n for n in z.namelist() if n.endswith('.csv')]
        dfs = []
        for name in csv_names:
            df = pd.read_csv(z.open(name), parse_dates=['valid_time'])
            df = df.drop(columns=['latitude', 'longitude'], errors='ignore')
            dfs.append(df.set_index('valid_time'))
        merged = pd.concat(dfs, axis=1).reset_index()

    merged.to_csv(out_csv, index=False)
    tmp_zip.unlink()

    return out_csv


# Delete old incomplete cache files (only had 2 vars)
import glob
old_cache = list(CACHE_DIR.glob('*.csv'))
if old_cache:
    sample_cols = pd.read_csv(old_cache[0], nrows=0).columns.tolist()
    if 'ssrd' not in sample_cols and 'surface_solar_radiation_downwards' not in sample_cols:
        print(f'Deleting {len(old_cache)} incomplete cache files (missing vars)...')
        for f in old_cache:
            f.unlink()

# Count already cached
cached_count = sum(1 for _, r in unique_gp.iterrows() if cache_path(r['grid_lat'], r['grid_lon']).exists())
print(f'Already cached: {cached_count} / {len(unique_gp)}')
print(f'To download:    {len(unique_gp) - cached_count}')

errors = []

for _, gp in tqdm(unique_gp.iterrows(), total=len(unique_gp), desc='Downloading grid points'):
    lat, lon = gp['grid_lat'], gp['grid_lon']
    if cache_path(lat, lon).exists():
        continue
    try:
        download_gridpoint(lat, lon)
    except Exception as e:
        errors.append((lat, lon, str(e)))
        print(f'  ERROR ({lat}, {lon}): {e}')
    time.sleep(SLEEP_BETWEEN)

cached_count = sum(1 for _, r in unique_gp.iterrows() if cache_path(r['grid_lat'], r['grid_lon']).exists())
print(f'\nGrid points cached: {cached_count} / {len(unique_gp)}')
print(f'Errors: {len(errors)}')

Deleting 386 incomplete cache files (missing vars)...
Already cached: 0 / 386
To download:    386



- The dataset presented here is a subset of selected parameters from the full [CDS ERA5 hourly data on single levels (1940–present)](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land?tab=overview). **Requirements for additional parameters may be considered**. Please raise your request with ECMWF Support [here](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202).
2026-03-15 16:41:56,204 INFO Request ID is 3adccfc2-fd31-46e9-babd-a2cbde536e08
2026-03-15 16:41:56,286 INFO status has been updated to accepted
2026-03-15 16:42:10,057 INFO status has been updated to running
2026-03-15 16:42:17,734 INFO status has been updated to successful

- The dataset presented here is a subset of selected parameters from the full [CDS ERA5 hourly data on single levels (1940–present)](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land?tab=overview). **Requirements for additional parameters may be considered**. Please raise your request with ECMWF Support [here](http


Grid points cached: 386 / 386
Errors: 0


## Step 2 — Build per-plant combined CSVs

In [13]:
def build_plant_csv(plant_id, grid_lat, grid_lon):
    """Read the cached grid point CSV, standardise columns, save as plant combined CSV."""
    out_dir  = ERA5_DIR / plant_id
    out_file = out_dir / f'{plant_id}_combined.csv'
    if out_file.exists():
        return

    src = cache_path(grid_lat, grid_lon)
    if not src.exists():
        return

    df = pd.read_csv(src)

    # Rename time column
    time_col = 'valid_time' if 'valid_time' in df.columns else 'time'
    df = df.rename(columns={time_col: 'timestamp'})
    df['timestamp'] = pd.to_datetime(df['timestamp']).dt.tz_localize(None)

    # Rename variable columns to short names
    short_map = {
        '2m_temperature'                    : 't2m',
        '2m_dewpoint_temperature'           : 'd2m',
        'surface_pressure'                  : 'sp',
        'total_precipitation'               : 'tp',
        'surface_solar_radiation_downwards'  : 'ssrd',
        'surface_thermal_radiation_downwards': 'strd',
    }
    for long_name, short_name in short_map.items():
        if long_name in df.columns:
            df = df.rename(columns={long_name: short_name})

    df['latitude']  = grid_lat
    df['longitude'] = grid_lon

    cols = ['timestamp', 'latitude', 'longitude', 'd2m', 't2m', 'sp', 'tp', 'ssrd', 'strd']
    available = [c for c in cols if c in df.columns]
    df = df[available].sort_values('timestamp').reset_index(drop=True)

    out_dir.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_file, index=False)


for _, row in tqdm(meta.iterrows(), total=len(meta), desc='Building plant CSVs'):
    build_plant_csv(row['plant_id'], row['grid_lat'], row['grid_lon'])

completed = [pid for pid in meta['plant_id'] if (ERA5_DIR / pid / f'{pid}_combined.csv').exists()]
print(f'\nPlants with ERA5: {len(completed)} / {len(meta)}')

Building plant CSVs: 100%|██████████| 593/593 [02:41<00:00,  3.68it/s]


Plants with ERA5: 593 / 593


## Step 3 — Validation

In [14]:
import random
sample_pid = random.choice(completed)
sample = pd.read_csv(ERA5_DIR / sample_pid / f'{sample_pid}_combined.csv')
print(f'Sample: {sample_pid}')
print(f'Shape:  {sample.shape}')
print(f'Period: {sample["timestamp"].min()} -> {sample["timestamp"].max()}')
print(f'Columns: {sample.columns.tolist()}')
print()
print(sample.describe().round(2))

Sample: plant_283
Shape:  (26304, 9)
Period: 2023-01-01 00:00:00 -> 2025-12-31 23:00:00
Columns: ['timestamp', 'latitude', 'longitude', 'd2m', 't2m', 'sp', 'tp', 'ssrd', 'strd']

       latitude  longitude       d2m       t2m        sp        tp  \
count   26304.0    26304.0  26304.00  26304.00  26304.00  26304.00   
mean       38.2       -6.6    281.90    290.42  96146.81      0.00   
std         0.0        0.0      4.44      8.28    555.35      0.00   
min        38.2       -6.6    264.15    271.71  93599.55     -0.00   
25%        38.2       -6.6    279.01    284.08  95830.54      0.00   
50%        38.2       -6.6    282.30    289.27  96104.06      0.00   
75%        38.2       -6.6    285.19    296.17  96459.97      0.00   
max        38.2       -6.6    293.11    313.69  97995.03      0.01   

             ssrd        strd  
count    26304.00    26304.00  
mean    743205.23  1172146.85  
std    1037910.99   147396.31  
min         -4.00   782749.00  
25%          0.00  1064043.73 